In [1]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv('./data/telco_churn_data.csv')
df
df.TotalCharges = df.TotalCharges.str.replace(' ', '0').astype(float)
df.drop('customerID', axis=1, inplace=True)

In [149]:
df.isna().sum()
df.duplicated().sum()


y = df['Churn']
df_object = df.select_dtypes(include='object').drop('Churn',axis=1)
df_numeric = df.select_dtypes(include='number')

df_object.nunique()
df[df['TotalCharges'] == ' ']
df[df['tenure'] == 0]

gt3_features = df_object.nunique()[df_object.nunique() >= 3].index.tolist()
df_object_gt3 = df_object[gt3_features]
df_object_lt3 = df_object.replace({"Yes": 1, "No": 0})

#gt3_features.pop()
gt3_features

df_object_gt3[gt3_features]


/tmp/ipykernel_30933/692727288.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_object = df.select_dtypes(include='object').drop('Churn',axis=1)


,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaymentMethod
0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Electronic check
1,No,DSL,Yes,No,Yes,No,No,No,One year,Mailed check
2,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Mailed check
3,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,Bank transfer (automatic)
4,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Electronic check
...,...,...,...,...,...,...,...,...,...,...
7038,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Mailed check
7039,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Credit card (automatic)
7040,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Electronic check
7041,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Mailed check


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import dill

def encoder_ohe(df, features):
  from sklearn.preprocessing import OneHotEncoder
  import pandas as pd
  ohe = OneHotEncoder()

  for f in features:
    encoded_f = ohe.fit_transform(df[[f]]).toarray()
    encoded_df = pd.DataFrame(encoded_f, columns=[f"{cat}_encoded" for cat in ohe.categories_[0]])
    df = pd.concat([df, encoded_df],axis=1).drop(f, axis=1)

  return (df, ohe)

df_encoded, ohe = encoder_ohe(df_object_gt3[gt3_features], gt3_features)
df_object[gt3_features].head(50)
df_encoded

ohe = OneHotEncoder()
encoded_features = ohe.fit(df_object_gt3[gt3_features])
#df_encoded2 = pd.DataFrame(encoded_features, columns=np.array(ohe.categories_).ravel())


with open('./data/encoder_ohe.pkl', 'wb') as encoder: 
  dill.dump(, encoder)


feature_labels = np.array(ohe.categories_)
feature_labels.ravel()
df_encoded2
ohe.get_feature_names_out(input_features=ohe.feature_names_in_)



In [35]:

df_encoded.columns
df_object[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection']].value_counts()

OnlineSecurity       OnlineBackup         DeviceProtection   
No internet service  No internet service  No internet service    1526
No                   No                   No                     1509
Yes                  Yes                  Yes                     693
No                   No                   Yes                     686
                     Yes                  No                      678
                                          Yes                     625
Yes                  No                   No                      475
                     Yes                  No                      433
                     No                   Yes                     418
Name: count, dtype: int64

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

X_train, X_test, y_train, y_test = train_test_split(pd.concat([df_numeric, df_encoded], axis=1), y, test_size=0.2, random_state=42)

dtc = DecisionTreeClassifier(max_depth=7)
rfc = RandomForestClassifier(max_depth=10, n_estimators=100, min_samples_split=3, random_state=42)
gbc = GradientBoostingClassifier(max_depth=7, n_estimators=1000, loss='exponential', learning_rate=0.05)

dtc.fit(X_train, y_train)
rfc.fit(X_train, y_train)
gbc.fit(X_train, y_train)

y_pred_dtc = dtc.predict(X_test)
y_pred_rfc = rfc.predict(X_test)
y_pred_gbc = gbc.predict(X_test)

print(f"DecisionTreeClassifier Accuracy: {accuracy_score(y_test, y_pred_dtc)}")
print(f"RandomForestClassifier Accuracy: {accuracy_score(y_test, y_pred_rfc)}")
print(f"GradientBoostingClassifier Accuracy: {accuracy_score(y_test, y_pred_gbc)}")





DecisionTreeClassifier Accuracy: 0.794889992902768
RandomForestClassifier Accuracy: 0.8097941802696949
GradientBoostingClassifier Accuracy: 0.7892122072391767


In [26]:
model_features = pd.concat([df_numeric, df_encoded], axis=1).columns
model_features
gt3_features
numeric_features = df_numeric.columns.tolist()

model_features
df_object_lt3
X_train.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'No_encoded', 'No phone service_encoded', 'Yes_encoded', 'DSL_encoded',
       'Fiber optic_encoded', 'No_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'No_encoded',
       'No internet service_encoded', 'Yes_encoded', 'Month-to-month_encoded',
       'One year_encoded', 'Two year_encoded',
       'Bank transfer (automatic)_encoded', 'Credit card (automatic)_encoded',
       'Electronic check_encoded', 'Mailed check_encoded'],
      dtype='str')

In [7]:
joblib.dump(rfc, "data/rfc_best_model.pkl")

['data/rfc_best_model.pkl']